# Advanced Landlab B: coupled event hydrology

**Duration:** 2 hours, including a 10-minute break  
**Prerequisite:** complete `01_landlab_fundamentals.ipynb`  
**Goal:** couple rainfall/runoff routing with Green-Ampt infiltration, close a
water budget, and test one parameter while holding the rest constant.

The advanced idea is **shared state**: two components read and modify the same
`surface_water__depth` field during each timestep.

## 1. Component contracts and shared fields

| Field | Location and units | Green-Ampt infiltration | Kinematic-wave flow |
| --- | --- | --- | --- |
| `topographic__elevation` | node, m | — | input |
| `surface_water__depth` | node, m | input **and** output | output |
| `soil_water_infiltration__depth` | node, m | input **and** output | — |
| `surface_water_inflow__discharge` | node, m³/s | — | output |

`surface_water__depth` is the coupling field. Overland flow adds rainfall and
redistributes surface water; infiltration removes available surface water and
adds the same depth to cumulative infiltration.

`KinwaveImplicitOverlandFlow.runoff_rate` is supplied in **mm/hr**, its timestep
is in **seconds**, and both water-depth fields are in **metres**. Landlab uses
these conventions but cannot detect arbitrary unit mistakes elsewhere in a
model.

> **Remember — component order defines the coupled approximation**
>
> For each timestep we first add and route rainfall with the kinematic-wave component, then infiltrate available surface water. This sequential update is operator splitting. Reversing the order changes which water is available to infiltrate during that step, especially when timesteps are large.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from landlab import RasterModelGrid, imshow_grid
from landlab.components import (
    KinwaveImplicitOverlandFlow,
    SoilInfiltrationGreenAmpt,
)
from landlab.io import esri_ascii

## 2. Load and preserve the supplied DEM

In [ ]:
from pathlib import Path

dem_path = Path("../data/bijou_gully_subset_5m_edit_dx_filled.asc")
with dem_path.open() as fp:
    base_grid = esri_ascii.load(fp, name="topographic__elevation", at="node")

# This explicit copy is the immutable starting topography for every run.
base_elevation = base_grid.at_node["topographic__elevation"].copy()
grid_shape = base_grid.shape
node_spacing = base_grid.dx

print(f"shape: {grid_shape}")
print(f"spacing: {node_spacing:.2f} m")
base_grid.imshow("topographic__elevation", cmap="terrain", colorbar_label="Elevation (m)")

> **Watch out — copy the starting topography**
>
> Assigning one NumPy array to several grids would make those grids share data. `base_elevation.copy()` gives every experiment independent initial state. The source DEM has also been pit-filled because this kinematic-wave method cannot route water reliably across unresolved pits and flats.

In [ ]:
def make_hydrology_grid():
    model_grid = RasterModelGrid(grid_shape, xy_spacing=node_spacing)
    elevation = model_grid.add_field(
        "topographic__elevation", base_elevation.copy(), at="node"
    )

    # Close every edge, then open one fixed-value outlet at the lowest node on
    # the bottom edge. This makes boundary discharge measurable.
    model_grid.set_closed_boundaries_at_grid_edges(
        right_is_closed=True,
        top_is_closed=True,
        left_is_closed=True,
        bottom_is_closed=True,
    )
    bottom = model_grid.nodes_at_bottom_edge
    outlet = bottom[np.argmin(elevation[bottom])]
    model_grid.status_at_node[outlet] = model_grid.BC_NODE_IS_FIXED_VALUE
    return model_grid, outlet


def depth_volume(model_grid, depth):
    # Raster core nodes each represent one cell of area dx * dy.
    return (
        np.sum(depth[model_grid.core_nodes])
        * model_grid.dx
        * model_grid.dy
    )


def water_budget(result):
    supplied = result["initial_surface_volume"] + result["rain_volume"]
    accounted = (
        result["final_surface_volume"]
        + result["infiltration_gain"]
        + result["outlet_volume"]
    )
    residual = supplied - accounted
    return supplied, accounted, residual

All edges are closed except one fixed-value outlet at the lowest node on the
bottom edge. This makes water leaving the domain measurable. A different outlet
choice would be a different scientific experiment.

## 3. Run and check infiltration alone

Green-Ampt represents a descending wetting front. The infiltration rate is
approximately

$$f=K\left(1+\frac{H_fM_d}{F}+\frac{H}{F}\right).$$

The cumulative infiltration field $F$ must start positive to avoid division by
zero. In this single-process test, every node has the same soil parameters and
surface-water depth, so the result should be spatially uniform. Topography does
not enter this component.

In [ ]:
infiltration_grid, _ = make_hydrology_grid()
surface = infiltration_grid.add_zeros("surface_water__depth", at="node")
surface[infiltration_grid.core_nodes] = 0.1  # 10 cm
infiltrated = infiltration_grid.add_full(
    "soil_water_infiltration__depth", 1.0e-4, at="node"
)

surface_initial = surface.copy()
infiltrated_initial = infiltrated.copy()
infiltration = SoilInfiltrationGreenAmpt(
    infiltration_grid, hydraulic_conductivity=5.0e-6
)

time_step = 10.0  # seconds
duration = 600.0  # seconds
for _ in range(int(duration / time_step)):
    infiltration.run_one_step(time_step)

surface_loss = depth_volume(infiltration_grid, surface_initial - surface)
infiltration_gain = depth_volume(
    infiltration_grid, infiltrated - infiltrated_initial
)
print(f"surface-water loss: {surface_loss:.6f} m³")
print(f"infiltration gain:  {infiltration_gain:.6f} m³")
print(f"difference:         {surface_loss - infiltration_gain:.3e} m³")

> **Model check — test conservation before coupling**
>
> For infiltration alone, surface-water loss should equal cumulative-infiltration gain to floating-point precision. If this simple budget fails, do not add another component yet.

## 4. Run and inspect overland flow alone

The kinematic-wave component adds rainfall, routes water downhill using surface
slope as a proxy for energy slope, stores water in
`surface_water__depth`, and reports discharge arriving at downstream nodes.
Water reaching the fixed outlet leaves the modeled core domain.

In [ ]:
print(KinwaveImplicitOverlandFlow.__init__.__doc__)

In [ ]:
flow_grid, flow_outlet = make_hydrology_grid()
flow_depth = flow_grid.add_zeros("surface_water__depth", at="node")
overland_flow = KinwaveImplicitOverlandFlow(
    flow_grid,
    runoff_rate=90.0,  # mm/hr
    roughness=0.1,
    depth_exp=5.0 / 3.0,
)

time_step = 10.0
storm_duration = 300.0
outlet_volume = 0.0
for _ in range(int(storm_duration / time_step)):
    overland_flow.run_one_step(time_step)
    outlet_volume += (
        flow_grid.at_node["surface_water_inflow__discharge"][flow_outlet]
        * time_step
    )

flow_grid.imshow(
    "surface_water__depth", cmap="Blues", colorbar_label="Water depth (m)"
)
print(f"water routed through outlet: {outlet_volume:.3f} m³")

<details>
<summary><strong>Optional theory — the locally implicit kinematic-wave update</strong></summary>

Conservation of water depth $H$ can be written as

$$\frac{\partial H}{\partial t}=R-\nabla\cdot\mathbf{q},$$

where $R$ is local runoff and $\mathbf{q}$ is unit discharge. The component
uses topographic slope and a Manning/Chezy-style relationship between depth and
discharge, then solves nodes from upstream to downstream. The `weight`
parameter controls how implicit the depth update is. This method requires
pit-filled topography because ground-surface slope substitutes for hydraulic
energy slope.
</details>

## Debugging checkpoint and break

With a partner, diagnose these cases:

1. `soil_water_infiltration__depth` starts at exactly zero.
2. Rainfall is converted to m/s before being passed as `runoff_rate`.
3. Two experiments attach the same `base_elevation` array without `.copy()`.
4. A water budget omits discharge through the outlet.
5. The two components are run once each instead of once per timestep.

Then take a 10-minute break.

## 5. Main challenge: couple rainfall, flow, and infiltration

Use the supplied values:

- initial surface water: 0.10 m on core nodes;
- initial cumulative infiltration: 0.01 m everywhere;
- hydraulic conductivity: $10^{-6}$ m/s;
- rainfall: 100 mm/hr;
- timestep: 10 s; and
- duration: 600 s.

Close the budget

> initial surface water + rainfall = final surface water + infiltration gain + outlet discharge.

The residual should be tiny relative to total supplied water.

In [ ]:
def run_coupled_hydrology(
    hydraulic_conductivity=1.0e-6,
    rainfall_rate=100.0,
    initial_surface_depth=0.1,
    initial_infiltration_depth=0.01,
    duration=600.0,
    time_step=10.0,
):
    model_grid, outlet = make_hydrology_grid()

    # TODO 1: create and initialize surface_water__depth.
    # Initialize core nodes with initial_surface_depth.

    # TODO 2: create soil_water_infiltration__depth and initialize every node
    # with a positive initial_infiltration_depth.

    # TODO 3: save independent copies of both initial fields.

    # TODO 4: instantiate SoilInfiltrationGreenAmpt and
    # KinwaveImplicitOverlandFlow. rainfall_rate is in mm/hr.

    outlet_volume = 0.0
    number_of_steps = int(duration / time_step)
    for _ in range(number_of_steps):
        # TODO 5: add and route rainfall with the kinematic-wave component.
        # TODO 6: accumulate discharge arriving at outlet during this step.
        # TODO 7: infiltrate available surface water.
        pass

    # TODO 8: calculate rain volume, final surface-water volume, and the
    # increase in infiltrated-water volume over core-node cells.

    # TODO 9: return a dictionary containing the grid, fields, snapshots,
    # volumes, outlet, and parameter values needed by the plotting function.
    return None


# After completing the TODOs, uncomment:
# coupled_result = run_coupled_hydrology()

<details>
<summary><strong>Hint 1 — required fields</strong></summary>

Create `surface_water__depth` with zeros and set core nodes to 0.10 m. Create
`soil_water_infiltration__depth` with a positive value at every node. Save
`.copy()` snapshots before instantiating the components.
</details>

<details>
<summary><strong>Hint 2 — loop order</strong></summary>

Call kinematic-wave flow first. Add the outlet's
`surface_water_inflow__discharge * time_step` to cumulative outlet volume. Then
call infiltration.
</details>

<details>
<summary><strong>Hint 3 — rainfall volume</strong></summary>

Convert mm/hr to m/s, multiply by duration, core-cell count, and cell area.
Compare that input plus initial surface storage against the three final terms.
</details>

In [ ]:
def plot_hydrology_result(result, title):
    model_grid = result["grid"]
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    fields = [
        (result["surface_initial"], "Initial surface water", "Blues"),
        (result["surface_final"], "Final surface water", "Blues"),
        (
            result["infiltration_initial"] * 1000.0,
            "Initial infiltrated depth (mm)",
            "GnBu",
        ),
        (
            result["infiltration_final"] * 1000.0,
            "Final infiltrated depth (mm)",
            "GnBu",
        ),
    ]
    for ax, (values, label, cmap) in zip(axes.flat, fields):
        plt.sca(ax)
        imshow_grid(model_grid, values, cmap=cmap, colorbar_label=label)
        ax.set_title(label)
    fig.suptitle(title)
    plt.tight_layout()


# Learners: run this after coupled_result has been created.
if "coupled_result" in globals() and coupled_result is not None:
    plot_hydrology_result(coupled_result, "Coupled rainfall–infiltration–flow")
    supplied, accounted, residual = water_budget(coupled_result)
    print(f"water supplied:  {supplied:.6f} m³")
    print(f"water accounted: {accounted:.6f} m³")
    print(f"budget residual: {residual:.3e} m³")
    print(f"relative error:  {residual / supplied:.3e}")
else:
    print("Complete the coupling challenge, then rerun this cell.")

> **Interpret maps together with the budget**
>
> A deep-water hotspot may reflect convergence, a depression, or a boundary choice. A closed budget says water was accounted for; it does not by itself prove that topography, parameters, or the kinematic-wave approximation are appropriate.

## 6. One-factor experiment

Predict how increasing hydraulic conductivity by one order of magnitude will
partition water among infiltration, surface storage, and outlet discharge.
Keep rainfall, initial fields, grid, boundaries, timestep, and duration fixed.

In [ ]:
low_conductivity = 1.0e-6
high_conductivity = 1.0e-5

# TODO: run one experiment for each conductivity while holding rainfall,
# initial water, duration, timestep, grid, and boundaries fixed.

# TODO: compare final surface storage, infiltration gain, and outlet volume.

**Prediction:**  
**Evidence:**  
**Interpretation:**

## 7. Project handoff (8 minutes)

Open `03_project_launch.ipynb`. A tractable project varies one control such as
hydraulic conductivity, rainfall, roughness, outlet location, or storm duration
and chooses a budget term or map diagnostic in advance.

Record field initializations, conversions, boundary status, component order,
timestep, duration, and Landlab version. Restart the kernel and run the final
model from top to bottom before sharing it.

## References

- Green, W. H., & Ampt, G. A. (1911), *Journal of Agricultural Science*, 4(1), 1–24.
- Julien, P. Y., Saghafian, B., & Ogden, F. L. (1995), *Journal of the American Water Resources Association*, 31, 523–536.
- Rengers, F. K. et al. (2016), *Water Resources Research*, 52, 6041–6061.